# AgroScan - Step 1: DVC pipeline + MLflow tracking (Colab)
Runs the pipeline in your GitHub repo on a Colab GPU: `unzip -> train -> evaluate`, tracked by DVC and MLflow.

**Before you start**
1. Runtime -> Change runtime type -> **T4 GPU**.
2. Colab Secrets (key icon, left): add `GITHUB_TOKEN` (a GitHub *fine-grained* token with **Contents: Read and write** on your AgroScan repo), notebook access ON.
3. Put your GitHub user and repo name in the next cell.
4. Do not share screenshots of the clone cell: its address contains your token.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import os, subprocess, glob

GITHUB_USER = 'YOUR_GITHUB_USERNAME'      # <-- change
REPO = 'agroscan'                          # <-- change if your repo has another name
GIT_NAME, GIT_EMAIL = 'Your Name', 'you@example.com'   # <-- change

DRIVE_ZIP = '/content/drive/MyDrive/AgroScan-v5/data/agroscan_prepared_v1.zip'     # made by Part 1
DVC_STORE = '/content/drive/MyDrive/AgroScan-v5/dvc-store'                          # the DVC "remote": a folder in your Drive
os.makedirs(DVC_STORE, exist_ok=True)
assert os.path.exists(DRIVE_ZIP), 'Part 1 zip not found in Drive'
import tensorflow as tf
print('GPU:', tf.config.list_physical_devices('GPU') or 'NONE - switch the runtime to GPU')

## 1. Get the code from GitHub and install DVC + MLflow

In [ ]:
token = userdata.get('GITHUB_TOKEN')
if not os.path.exists('/content/agroscan'):
    !git clone -q https://{token}@github.com/{GITHUB_USER}/{REPO}.git /content/agroscan
os.chdir('/content/agroscan')
!git config user.email "{GIT_EMAIL}"
!git config user.name "{GIT_NAME}"
!pip install -q dvc mlflow pyyaml
print('CHECKPOINT: repo files ->', sorted(os.listdir('.')))

## 2. Set up DVC and bring in the dataset
First time: the zip is copied from Drive and registered with DVC (`dvc add`). Next sessions: it is pulled from the DVC remote.

In [ ]:
if not os.path.exists('.dvc'):
    !dvc init -q
    !git add .dvc .dvcignore
    !git commit -q -m "Initialize DVC"
!dvc remote add -d -f storage {DVC_STORE}

if os.path.exists('data/agroscan_prepared_v1.zip.dvc'):
    !dvc pull data/agroscan_prepared_v1.zip.dvc
else:
    !cp {DRIVE_ZIP} data/agroscan_prepared_v1.zip
    !dvc add data/agroscan_prepared_v1.zip
    !git add data/agroscan_prepared_v1.zip.dvc data/.gitignore .dvc/config
    !git commit -q -m "Track prepared dataset v1 with DVC"
    !dvc push
print('CHECKPOINT: you should see data/agroscan_prepared_v1.zip below')
!ls -lh data

## 3. Look at the settings (params.yaml)

In [ ]:
print(open('params.yaml').read())

## 4. Run the pipeline (the long step: roughly 10-20 minutes)
`dvc repro` runs `unzip -> train -> evaluate`. If nothing changed since the last run, it skips stages that are up to date.

In [ ]:
!dvc repro
print('CHECKPOINT: you should see "Running stage unzip / train / evaluate" and a line "registered model version"')

## 5. Results (screenshot these)

In [ ]:
!dvc metrics show
from IPython.display import Image, display
for f in sorted(glob.glob('reports/figures/*.png')):
    print(f); display(Image(f, width=520))

## 6. Save everything: DVC remote + Git
Big files go to the DVC remote (your Drive folder), small files and the pipeline state go to GitHub.

In [ ]:
!dvc push
!git add dvc.lock dvc.yaml params.yaml reports/train_metrics.json reports/test_metrics.json models/.gitignore
!git commit -q -m "Pipeline run: train and evaluate"
!git push -q origin HEAD
print('CHECKPOINT: open your GitHub repo - you should see the new commit with dvc.lock and the metrics files')

## 7. Export the MLflow store to Drive (to open its dashboard on your laptop)

In [ ]:
EXPORT = '/content/drive/MyDrive/AgroScan-v5/mlflow_export'
os.makedirs(EXPORT, exist_ok=True)
!cp -r mlflow.db mlartifacts {EXPORT}/
print('Exported to', EXPORT)
print('On your laptop: copy that folder, then follow "View MLflow on your laptop" in the repo README.')